In [ ]:
'''
KAGGLE HOME CREDIT DEFAULT RISK COMPETITION
Adapted from one of the models used in 7th place solution ensemble.
For more details about our solution please check this discussion:
https://www.kaggle.com/c/home-credit-default-risk/discussion/64580

Another similar version is also available at GitHub:
https://github.com/js-aguiar/home-credit-default-competition

This model uses LightGBM with goss and label encode for the application's categorical features.
Other tables are using one-hot encode with mean, sum and a few different functions to aggregate.
The main idea was to add more time related features like last application and last X months aggregations.
There are also aggregations for specific loan types and status as well as ratios between tables.
Configurations are in line 785
'''

import os
import gc
import time
import numpy as np
import pandas as pd
from contextlib import contextmanager
import multiprocessing as mp
from functools import partial
from scipy.stats import kurtosis, iqr, skew
from lightgbm import LGBMClassifier
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.simmplefilter(action='ignore', category=FutureWarning)

def main(debug=False):
    num_rows = 30000 if debug else None
    with timer('application_train and application_test'):
        df = get_train_test(DATA_DIRECTORY, num_rows=num_rows)
        print('Application dataframe shape: ', df.shape)
    with timer('Bureau and bureau_balance data'):
        bureau_df = get_bureau(DATA_DIRECTORY, num_rows=num_rows)
        df = pd.merge(df, bureau_df, on='SK_ID_CURR', how='left')
        print('Bureau dataframe shape: ', bureau_df.shape)
        del bureau_df; gc.collect()
    with timer('previous_application'):
        prev_df = get_previous_applications(DATA_DIRECTORY, num_rows)
        df = pd.merge(df, prev_df, on='SK_ID_CURR', how='left')
        print('Previous dataframe shape: ', prev_df.shape)
        del prev_df; gc.collect()
    with timer('previous ')